In [ ]:
from pynq import Overlay
from pynq.lib.video import *
from PIL import Image
import numpy as np

# Load the base overlay (contains HDMI pipeline)
overlay = Overlay("/home/xilinx/pynq/overlays/base/base.bit")

# Configure HDMI output
hdmi_out = overlay.video.hdmi_out
mode = VideoMode(1920, 1080, 24)  # resolution: 640x480, 24-bit RGB
hdmi_out.configure(mode)
hdmi_out.start()

# Load and resize image to match HDMI resolution
img = Image.open("pokebg.jpg").resize((1920, 1080))
img_np = np.array(img)

# Create frame buffer
frame = hdmi_out.newframe()

# Copy image into frame
frame[:] = img_np

# Display on HDMI
hdmi_out.writeframe(frame)

print("Image displayed on HDMI.")

In [ ]:
print(hdmi_out.mode)

In [ ]:
from pynq import Overlay
from pynq.lib.video import *
from PIL import Image
import numpy as np

# Load base overlay (HDMI pipeline)
overlay = Overlay("/home/xilinx/pynq/overlays/base/base.bit")

hdmi_out = overlay.video.hdmi_out
mode = VideoMode(1920, 1080, 24)
hdmi_out.configure(mode)
hdmi_out.start()

WIDTH = 1920
HEIGHT = 1080

# Load images
background = Image.open("pokebg.jpg").resize((WIDTH, HEIGHT)).convert("RGBA")
image2 = Image.open("pika1.png").convert("RGBA")  # PNG supports transparency

# Position where overlay will appear
x = 400
y = 200

# Combine images
combined = background.copy()
combined.paste(image2, (x, y), image2)

img_np = np.array(combined.convert("RGB"), dtype=np.uint8)

frame = hdmi_out.newframe()
frame[:, :, :] = img_np
hdmi_out.writeframe(frame)

print("Overlay displayed")

In [ ]:
print(hdmi_out.mode)

In [ ]:
print("HDMI mode:", hdmi_out.mode)
print("Image shape:", img_np.shape)

In [ ]:
# Imports
from pynq import Overlay
from pynq.lib.video import *
from PIL import Image
import numpy as np

# Load base overlay (HDMI pipeline)
overlay = Overlay("/home/xilinx/pynq/overlays/base/base.bit")

# Configure HDMI output for 32-bit color
hdmi_out = overlay.video.hdmi_out
mode = VideoMode(1920, 1080, 32, 60)  # 32 bpp, 60 fps
hdmi_out.configure(mode)
hdmi_out.start()

# Load images and convert to RGBA (32-bit)
background = Image.open("pokebg.jpg").resize((1920, 1080)).convert("RGBA")
overlay_img = Image.open("pika1.png").resize((172,148)).convert("RGBA")  # PNG with transparency

# Position of overlay
x = 1330
y = 520

# Combine images
combined = background.copy()
combined.paste(overlay_img, (x, y), overlay_img)  # preserve transparency

# Convert to NumPy array for HDMI framebuffer
frame_np = np.array(combined, dtype=np.uint8)

# Create framebuffer and write the combined image
frame = hdmi_out.newframe()
frame[:] = frame_np
hdmi_out.writeframe(frame)

print("Background with overlay displayed on HDMI.")

In [ ]:
# Imports
from pynq import Overlay
from pynq.lib.video import *
from PIL import Image
import numpy as np
import time

# Load base overlay (HDMI pipeline)
overlay = Overlay("/home/xilinx/pynq/overlays/base/base.bit")

# Configure HDMI output for 32-bit color
hdmi_out = overlay.video.hdmi_out
mode = VideoMode(1920, 1080, 32, 60)  # 32 bpp, 60 fps
hdmi_out.configure(mode)
hdmi_out.start()

# Load images and convert to RGBA
background = Image.open("pokebg.jpg").resize((1920, 1080)).convert("RGBA")
overlay_img = Image.open("pika1.png").resize((172,148)).convert("RGBA")  # PNG with transparency

# Shift entire scene left by 200 pixels (optional)
frame_np = np.array(background, dtype=np.uint8)
shift_amount = 0
shifted_frame = np.zeros_like(frame_np)
shifted_frame[:, 0:1920-shift_amount, :] = frame_np[:, shift_amount:1920, :]
background = Image.fromarray(shifted_frame)

# Overlay initial position
x = 1330
y = 520
speed = 30  # pixels per frame
direction = 1  # 1 = moving right, -1 = moving left

while True:
    # Combine background and overlay at current position
    combined = background.copy()
    combined.paste(overlay_img, (x, y), overlay_img)
    
    # Convert to NumPy array and send to HDMI
    frame_np = np.array(combined, dtype=np.uint8)
    frame = hdmi_out.newframe()
    frame[:] = frame_np
    hdmi_out.writeframe(frame)
    
    # Update overlay position
    x += speed * direction
    
    # Bounce at edges
    if direction >= 0:
        direction = -1
    elif direction <= 0:
        direction = 1
    
    # Small delay to control speed (~30 FPS)
    time.sleep(0.03)

In [ ]:
#Pika + Mew left to right
from pynq import Overlay
from pynq.lib.video import *
from PIL import Image
import numpy as np
import time

# Load base overlay (HDMI pipeline)
overlay = Overlay("/home/xilinx/pynq/overlays/base/base.bit")

# Configure HDMI output
hdmi_out = overlay.video.hdmi_out
mode = VideoMode(1920, 1080, 32, 60)
hdmi_out.configure(mode)
hdmi_out.start()

# Load background
background = Image.open("pokebg.jpg").resize((1920, 1080)).convert("RGBA")

# Load overlays
pikachu = Image.open("pika1.png").resize((172,148)).convert("RGBA")
charmander = Image.open("mew2.png").resize((300,200)).convert("RGBA")

# Overlay initial positions and speeds
x_pika, y_pika, dir_pika, speed_pika = 1330, 520, 1, 10
x_char, y_char, dir_char, speed_char = 300, 880, -1, 7

while True:
    # Start with background
    combined = background.copy()
    
    # Paste overlays
    combined.paste(pikachu, (x_pika, y_pika), pikachu)
    combined.paste(charmander, (x_char, y_char), charmander)
    
    # Convert to NumPy array and send to HDMI
    frame_np = np.array(combined, dtype=np.uint8)
    frame = hdmi_out.newframe()
    frame[:] = frame_np
    hdmi_out.writeframe(frame)
    
    # Update positions
    x_pika += speed_pika * dir_pika
    x_char += speed_char * dir_char
    
    # Bounce Pikachu at screen edges
    if dir_pika >= 0:
        dir_pika = -1
        dir_char = -1
    elif dir_pika <= 0:
        dir_pika = 1
        dir_char = 1
    
    # Small delay for ~30 FPS
    time.sleep(0.03)

In [ ]:
from pynq import Overlay
from pynq.lib.video import *
from PIL import Image
import numpy as np
import time

# Load base overlay
overlay = Overlay("/home/xilinx/pynq/overlays/base/base.bit")

# HDMI configuration
hdmi_out = overlay.video.hdmi_out
mode = VideoMode(1920, 1080, 32, 60)
hdmi_out.configure(mode)
hdmi_out.start()

# Load background and overlays
background = Image.open("pokebg.jpg").resize((1920, 1080)).convert("RGBA")
pikachu = Image.open("pika1.png").resize((192,128)).convert("RGBA")
charmander = Image.open("char1.png").resize((192,128)).convert("RGBA")

# Overlay positions
x_pika, y_pika, dir_pika, speed_pika = 500, 500, 1, 10
x_char, y_char, dir_char, speed_char = 1200, 600, -1, 7

# Initialize a single framebuffer once
frame = hdmi_out.newframe()
frame_np = np.array(background, dtype=np.uint8)

while True:
    # Start from a copy of the background
    combined = background.copy()
    
    # Paste overlays
    combined.paste(pikachu, (x_pika, y_pika), pikachu)
    combined.paste(charmander, (x_char, y_char), charmander)
    
    # Copy into HDMI framebuffer once fully prepared
    frame[:] = np.array(combined, dtype=np.uint8)
    hdmi_out.writeframe(frame)
    
    # Update positions
    x_pika += speed_pika * dir_pika
    x_char += speed_char * dir_char
    
    # Bounce logic
    if x_pika + pikachu.width >= 1920 or x_pika <= 0:
        dir_pika *= -1
    if x_char + charmander.width >= 1920 or x_char <= 0:
        dir_char *= -1
    
    time.sleep(0.03)

In [ ]:
#asyncio version + attempt at hp-bar
from pynq import Overlay
from pynq.lib.video import *
from PIL import Image
import numpy as np
import time
import asyncio

# Load base overlay (HDMI pipeline)
overlay = Overlay("/home/xilinx/pynq/overlays/base/base.bit")

# Configure HDMI output
hdmi_out = overlay.video.hdmi_out
mode = VideoMode(1920, 1080, 32, 60)
hdmi_out.configure(mode)
hdmi_out.start()

# Load background
background = Image.open("pokebg.jpg").resize((1920, 1080)).convert("RGBA")

# Load overlays
pikachu = Image.open("pika1.png").resize((172,148)).convert("RGBA")
charmander = Image.open("mew2.png").resize((300,200)).convert("RGBA")
hp0 = Image.open("HP0.png").resize((150,100)).convert("RGBA")
hp1 = Image.open("HP1.png").resize((150,100)).convert("RGBA")

# Overlay initial positions and speeds
x_pika, y_pika, dir_pika, speed_pika = 1330, 520, 1, 10
x_char, y_char, dir_char, speed_char = 300, 880, -1, 7
x_hp0, y_hp0, x_hp1, y_hp1 = 100, 100, 100, 100
hp_percentage0, hp_percentage1 = 100, 100
running = True
hp0_changed = False
hp1_changed = False
def hp0_change():
    global hp_resized, hp0_changed
    hp0_width = 50 + 100 * hp0_percentage  # e.g., from 50 to 150 pixels
    hp_height = 100  # fixed height
    hp0_resized = hp0.resize((int(hp0_width), hp_height)).convert("RGBA")
    hp0_changed = False
def change_hp0(int a):
    global hp_percentage0
    hp_percentage0 = a
    hp0_changed = True
async def animation_loop():
    global x_pika, x_char, dir_pika, dir_char
    frame = hdmi_out.newframe()
    while True:
        if running: 
            x_pika += speed_pika * dir_pika
            x_char += speed_char * dir_char
            if dir_pika >= 0:
                dir_pika = -1
                dir_char = -1
            elif dir_pika <= 0:
                dir_pika = 1
                dir_char = 1
            combined = background.copy()
            combined.paste(pikachu, (int(x_pika), y_pika), pikachu)
            combined.paste(charmander, (int(x_char), y_char), charmander)
            if hp0_changed:
                hp_change()
            # Paste at fixed coordinates
            combined.paste(hp0_resized, (x_hp0, y_hp0), hp0_resized)
            #combined.paste(hp1 if hp else hp0, (x_hp0, y_hp0), hp1 if hp else hp0)

            frame[:] = np.array(combined, dtype=np.uint8)
            hdmi_out.writeframe(frame)
        await asyncio.sleep(0.016)

In [ ]:
#asyncio version
from pynq import Overlay, MMIO
from pynq.lib.video import *
from PIL import Image
import numpy as np
import time
import asyncio

# Load base overlay (HDMI pipeline)
overlay = Overlay("/home/xilinx/pynq/overlays/base/base.bit")
# Configure HDMI output
hdmi_out = overlay.video.hdmi_out
mode = VideoMode(1920, 1080, 32, 60)
hdmi_out.configure(mode)
hdmi_out.start()

# Load background
background = Image.open("pokebg.jpg").resize((1920, 1080)).convert("RGBA")

# Load overlays
pikachu = Image.open("pika1.png").resize((172,148)).convert("RGBA")
charmander = Image.open("mew2.png").resize((300,200)).convert("RGBA")
hp0 = Image.open("hp.png").resize((150,100)).convert("RGBA")
hp1 = Image.open("HP1.png").resize((150,100)).convert("RGBA")

# Overlay initial positions and speeds
x_pika, y_pika, dir_pika, speed_pika = 1330, 520, 1, 10
x_char, y_char, dir_char, speed_char = 300, 880, -1, 7
x_hp0, y_hp0, x_hp1, y_hp1 = 100, 100, 100, 100
running = True
hp = True
hp0_change = False
hp0_percentage = 100
def update_hp0():
    global hp0, hp0_change
    hp_width = int(50 + 100 * hp0_percentage / 100)  # width scales from 50 to 150
    hp0 = hp0.resize((hp_width, 100)).convert("RGBA")
    hp0_change = False
    print("update")
def change_hp0(new_percentage):
    global hp0_percentage, hp0_change
    hp0_percentage = max(0, min(100, new_percentage))  # clamp 0-100%
    hp0_change = True
    print("change.")

async def animation_loop():
    global x_pika, x_char, dir_pika, dir_char
    frame = hdmi_out.newframe()
    while True:
        if running: 
            x_pika += speed_pika * dir_pika
            x_char += speed_char * dir_char
            if dir_pika >= 0:
                dir_pika = -1
                dir_char = -1
            elif dir_pika <= 0:
                dir_pika = 1
                dir_char = 1
            combined = background.copy()
            combined.paste(pikachu, (int(x_pika), y_pika), pikachu)
            combined.paste(charmander, (int(x_char), y_char), charmander)
            if hp0_change:
                update_hp0()
            combined.paste(hp0, (x_hp0, y_hp0), hp0)

            frame[:] = np.array(combined, dtype=np.uint8)
            hdmi_out.writeframe(frame)
        await asyncio.sleep(0.016)

In [ ]:
task = asyncio.create_task(animation_loop())

In [ ]:
running = False

In [ ]:
running = True

In [ ]:
hp = False

In [ ]:
hp = True

In [ ]:
change_hp0(100)

In [ ]:
from pynq import Overlay, MMIO
import time

BIT = "/home/xilinx/jupyter_notebooks/battle_block.bit"
ol = Overlay(BIT)

ip_name = [k for k in ol.ip_dict.keys() if "pokemon" in k.lower() or "battle" in k.lower()][0]
base = ol.ip_dict[ip_name]["phys_addr"]
mmio = MMIO(base, 0x1000)

CTRL=0x00
S0=0x04; S1=0x08; S2=0x0C; S3=0x10
O0=0x14; O1=0x18; O2=0x1C; O3=0x20

def start_and_wait():
    mmio.write(CTRL, 0x2)      # clear done (bit1)
    mmio.write(CTRL, 0x1)      # start (bit0)
    for i in range(200000):
        if (mmio.read(CTRL) >> 1) & 1:
            return True
    return False

mmio.write(S0, 0x00000000)
mmio.write(S1, 0x00000010)
mmio.write(S2, 0x00000053)
mmio.write(S3, 0x00000064)

ok = start_and_wait()
print("done:", ok, "CTRL:", hex(mmio.read(CTRL)))
#Read register values
print("OUT:", [hex(mmio.read(x)) for x in (O0,O1,O2,O3)])

In [ ]:
#change_hp0(int) function from asyncio version don't use for testing battle_blcok.bit overlay
change_hp0(mmio.read(O0))

In [ ]:
change_hp0(mmio.read(O1))

In [ ]:
change_hp0(mmio.read(O2))

In [ ]:
change_hp0(mmio.read(O3))

In [ ]:
print(hex(mmio.read(O3)))

In [ ]:
print("OUT:", [hex(mmio.read(x)) for x in (O0,O1,O2,O3)])

In [ ]:
#asyncio version
from pynq import Overlay, MMIO
from pynq.lib.video import *
from PIL import Image
import numpy as np
import time
import asyncio

BIT = "/home/xilinx/jupyter_notebooks/battle_block.bit"
ol = Overlay(BIT)

ip_name = [k for k in ol.ip_dict.keys() if "pokemon" in k.lower() or "battle" in k.lower()][0]
base = ol.ip_dict[ip_name]["phys_addr"]
mmio = MMIO(base, 0x1000)

CTRL=0x00
S0=0x04; S1=0x08; S2=0x0C; S3=0x10
O0=0x14; O1=0x18; O2=0x1C; O3=0x20

def start_and_wait():
    mmio.write(CTRL, 0x2)      # clear done (bit1)
    mmio.write(CTRL, 0x1)      # start (bit0)
    for i in range(200000):
        if (mmio.read(CTRL) >> 1) & 1:
            return True
    return False

mmio.write(S0, 0x00000000)
mmio.write(S1, 0x00000010)
mmio.write(S2, 0x00000053)
mmio.write(S3, 0x00000064)

ok = start_and_wait()
print("done:", ok, "CTRL:", hex(mmio.read(CTRL)))
print("OUT:", [hex(mmio.read(x)) for x in (O0,O1,O2,O3)])

# Load base overlay (HDMI pipeline)
overlay = Overlay("/home/xilinx/pynq/overlays/base/base.bit")
#This should be the same as the value from before but it changes because we have a second overlay call
#Combine battle_block.bit and base.bit to have only 1 overlay call
print("OUT:", [hex(mmio.read(x)) for x in (O0,O1,O2,O3)])
# Configure HDMI output
hdmi_out = overlay.video.hdmi_out
mode = VideoMode(1920, 1080, 32, 60)
hdmi_out.configure(mode)
hdmi_out.start()

# Load background
background = Image.open("pokebg.jpg").resize((1920, 1080)).convert("RGBA")

# Load overlays
pikachu = Image.open("pika1.png").resize((172,148)).convert("RGBA")
charmander = Image.open("mew2.png").resize((300,200)).convert("RGBA")
hp0 = Image.open("hp.png").resize((150,100)).convert("RGBA")
hp1 = Image.open("HP1.png").resize((150,100)).convert("RGBA")

# Overlay initial positions and speeds
x_pika, y_pika, dir_pika, speed_pika = 1330, 520, 1, 10
x_char, y_char, dir_char, speed_char = 300, 880, -1, 7
x_hp0, y_hp0, x_hp1, y_hp1 = 100, 100, 100, 100
running = True
hp = True
hp0_change = False
hp0_percentage = 100
def update_hp0():
    global hp0, hp0_change
    hp_width = int(50 + 100 * hp0_percentage / 100)  # width scales from 50 to 150
    hp0 = hp0.resize((hp_width, 100)).convert("RGBA")
    hp0_change = False
    print("update")
def change_hp0(new_percentage):
    global hp0_percentage, hp0_change
    hp0_percentage = max(0, min(100, new_percentage))  # clamp 0-100%
    hp0_change = True
    print("change.")

async def animation_loop():
    global x_pika, x_char, dir_pika, dir_char
    frame = hdmi_out.newframe()
    while True:
        if running: 
            x_pika += speed_pika * dir_pika
            x_char += speed_char * dir_char
            if dir_pika >= 0:
                dir_pika = -1
                dir_char = -1
            elif dir_pika <= 0:
                dir_pika = 1
                dir_char = 1
            combined = background.copy()
            combined.paste(pikachu, (int(x_pika), y_pika), pikachu)
            combined.paste(charmander, (int(x_char), y_char), charmander)
            if hp0_change:
                update_hp0()
            combined.paste(hp0, (x_hp0, y_hp0), hp0)

            frame[:] = np.array(combined, dtype=np.uint8)
            hdmi_out.writeframe(frame)
        await asyncio.sleep(0.016)

In [ ]:
from PIL import Image
import numpy as np

# ==============================
# USER SETTINGS
# ==============================
input_png  = "pokebg.jpg"     # uploaded PNG file
output_coe = "image.coe"     # output file name
width  = 320                 # BRAM image width
height = 240                 # BRAM image height
# ==============================

# Open image
img = Image.open(input_png).convert("RGB")

# Resize to desired resolution
img = img.resize((width, height), Image.LANCZOS)

# Convert to numpy array
img_array = np.array(img)

# Flatten into 1D array (row-major)
pixels = img_array.reshape(-1, 3)

# Convert each pixel to 24-bit hex string
hex_pixels = []
for r, g, b in pixels:
    hex_value = (r << 16) | (g << 8) | b
    hex_pixels.append(f"{hex_value:06X}")

# Write COE file
with open(output_coe, "w") as f:
    f.write("memory_initialization_radix=16;\n")
    f.write("memory_initialization_vector=\n")
    for i, value in enumerate(hex_pixels):
        if i == len(hex_pixels) - 1:
            f.write(value + ";\n")
        else:
            f.write(value + ",\n")

print("COE file generated successfully!")
print(f"Resolution: {width}x{height}")
print(f"Total pixels: {width*height}")
print(f"Output file: {output_coe}")

In [ ]:
#asyncio version
from pynq import Overlay, MMIO
from pynq.lib.video import *
from PIL import Image
import numpy as np
import time
import asyncio

BIT = "/home/xilinx/jupyter_notebooks/battle_block.bit"
ol = Overlay(BIT)

ip_name = [k for k in ol.ip_dict.keys() if "pokemon" in k.lower() or "battle" in k.lower()][0]
base = ol.ip_dict[ip_name]["phys_addr"]
mmio = MMIO(base, 0x1000)

CTRL=0x00
S0=0x04; S1=0x08; S2=0x0C; S3=0x10
O0=0x14; O1=0x18; O2=0x1C; O3=0x20

def start_and_wait():
    mmio.write(CTRL, 0x2)      # clear done (bit1)
    mmio.write(CTRL, 0x1)      # start (bit0)
    for i in range(200000):
        if (mmio.read(CTRL) >> 1) & 1:
            return True
    return False

mmio.write(S0, 0x00000000)
mmio.write(S1, 0x00000010)
mmio.write(S2, 0x00000053)
mmio.write(S3, 0x00000064)

ok = start_and_wait()
print("done:", ok, "CTRL:", hex(mmio.read(CTRL)))
print("OUT:", [hex(mmio.read(x)) for x in (O0,O1,O2,O3)])

# Load base overlay (HDMI pipeline)
#overlay = Overlay("/home/xilinx/pynq/overlays/base/base.bit")
#This should be the same as the value from before but it changes because we have a second overlay call
#Combine battle_block.bit and base.bit to have only 1 overlay call
print("OUT:", [hex(mmio.read(x)) for x in (O0,O1,O2,O3)])
# Configure HDMI output
hdmi_out = ol.video.hdmi_out
mode = VideoMode(1920, 1080, 32, 60)
hdmi_out.configure(mode)
hdmi_out.start()

# Load background
background = Image.open("pokebg.jpg").resize((1920, 1080)).convert("RGBA")

# Load overlays
pikachu = Image.open("pika1.png").resize((172,148)).convert("RGBA")
charmander = Image.open("mew2.png").resize((300,200)).convert("RGBA")
hp0 = Image.open("hp.png").resize((150,100)).convert("RGBA")
hp1 = Image.open("HP1.png").resize((150,100)).convert("RGBA")

# Overlay initial positions and speeds
x_pika, y_pika, dir_pika, speed_pika = 1330, 520, 1, 10
x_char, y_char, dir_char, speed_char = 300, 880, -1, 7
x_hp0, y_hp0, x_hp1, y_hp1 = 100, 100, 100, 100
running = True
hp = True
hp0_change = False
hp0_percentage = 100
def update_hp0():
    global hp0, hp0_change
    hp_width = int(50 + 100 * hp0_percentage / 100)  # width scales from 50 to 150
    hp0 = hp0.resize((hp_width, 100)).convert("RGBA")
    hp0_change = False
    print("update")
def change_hp0(new_percentage):
    global hp0_percentage, hp0_change
    hp0_percentage = max(0, min(100, new_percentage))  # clamp 0-100%
    hp0_change = True
    print("change.")

async def animation_loop():
    global x_pika, x_char, dir_pika, dir_char
    frame = hdmi_out.newframe()
    while True:
        if running: 
            x_pika += speed_pika * dir_pika
            x_char += speed_char * dir_char
            if dir_pika >= 0:
                dir_pika = -1
                dir_char = -1
            elif dir_pika <= 0:
                dir_pika = 1
                dir_char = 1
            combined = background.copy()
            combined.paste(pikachu, (int(x_pika), y_pika), pikachu)
            combined.paste(charmander, (int(x_char), y_char), charmander)
            if hp0_change:
                update_hp0()
            combined.paste(hp0, (x_hp0, y_hp0), hp0)

            frame[:] = np.array(combined, dtype=np.uint8)
            hdmi_out.writeframe(frame)
        await asyncio.sleep(0.016)

In [ ]:
# asyncio latency-measured version
from pynq import Overlay
from pynq.lib.video import *
from PIL import Image
import numpy as np
import time
import asyncio

# -------------------------
# CONFIG
# -------------------------
PROFILE = False   # set True to see detailed breakdown
TARGET_FPS = 60
FRAME_TIME = 1.0 / TARGET_FPS

# -------------------------
# LOAD OVERLAY
# -------------------------
overlay = Overlay("/home/xilinx/pynq/overlays/base/base.bit")

hdmi_out = overlay.video.hdmi_out
mode = VideoMode(1920, 1080, 32, 60)
hdmi_out.configure(mode)
hdmi_out.start()

# -------------------------
# LOAD IMAGES
# -------------------------
background = Image.open("pokebg.jpg").resize((1920, 1080)).convert("RGBA")

pikachu = Image.open("pika1.png").resize((172,148)).convert("RGBA")
charmander = Image.open("mew2.png").resize((300,200)).convert("RGBA")

hp0_base = Image.open("hp.png").resize((150,100)).convert("RGBA")
hp0 = hp0_base.copy()

# -------------------------
# STATE
# -------------------------
x_pika, y_pika, dir_pika, speed_pika = 1330, 520, 1, 10
x_char, y_char, dir_char, speed_char = 300, 880, -1, 7

x_hp0, y_hp0 = 100, 100

running = True

hp0_percentage = 100
hp0_change = False
hp0_timestamp = None   # for latency measurement

# -------------------------
# HP FUNCTIONS
# -------------------------
def update_hp0():
    global hp0, hp0_change

    hp_width = int(50 + 100 * hp0_percentage / 100)
    hp0 = hp0_base.resize((hp_width, 100)).convert("RGBA")

    hp0_change = False


def change_hp0(new_percentage):
    global hp0_percentage, hp0_change, hp0_timestamp

    hp0_percentage = max(0, min(100, new_percentage))
    hp0_change = True
    hp0_timestamp = time.perf_counter()


# -------------------------
# MAIN LOOP
# -------------------------
async def animation_loop():
    global x_pika, x_char, dir_pika, dir_char

    while True:
        loop_start = time.perf_counter()

        if running:
            # -------------------------
            # UPDATE POSITIONS
            # -------------------------
            x_pika += speed_pika * dir_pika
            x_char += speed_char * dir_char

            if dir_pika >= 0:
                dir_pika = -1
                dir_char = -1
            else:
                dir_pika = 1
                dir_char = 1

            # -------------------------
            # RENDER FRAME
            # -------------------------
            t0 = time.perf_counter()

            combined = background.copy()

            t1 = time.perf_counter()

            combined.paste(pikachu, (int(x_pika), y_pika), pikachu)
            combined.paste(charmander, (int(x_char), y_char), charmander)

            t2 = time.perf_counter()

            if hp0_change:
                update_hp0()

                if hp0_timestamp is not None:
                    latency = (time.perf_counter() - hp0_timestamp) * 1000
                    print(f"[LATENCY] HP update → frame: {latency:.2f} ms")

            combined.paste(hp0, (x_hp0, y_hp0), hp0)

            t3 = time.perf_counter()

            frame = hdmi_out.newframe()
            frame[:] = np.array(combined, dtype=np.uint8)

            t4 = time.perf_counter()

            hdmi_out.writeframe(frame)

            t5 = time.perf_counter()

            # -------------------------
            # PROFILING OUTPUT
            # -------------------------
            if PROFILE:
                print(f"""
copy bg:   {(t1 - t0)*1000:.2f} ms
paste:     {(t2 - t1)*1000:.2f} ms
hp update: {(t3 - t2)*1000:.2f} ms
to numpy:  {(t4 - t3)*1000:.2f} ms
write:     {(t5 - t4)*1000:.2f} ms
""")

        # -------------------------
        # FRAME TIMING
        # -------------------------
        loop_end = time.perf_counter()
        elapsed = loop_end - loop_start

        fps = 1.0 / elapsed if elapsed > 0 else 0
        print(f"[FRAME] {elapsed*1000:.2f} ms ({fps:.1f} FPS)")

        sleep_time = max(0, FRAME_TIME - elapsed)
        await asyncio.sleep(sleep_time)


In [ ]:
async def main():
    await animation_loop()

asyncio.run(main())